# Prediksi Hujan 12 Jam ke Depan (LSTM)
Input: data cuaca **per jam selama 7 hari terakhir** (suhu, kelembapan, tekanan, awan, angin, hujan) di suatu lokasi.
Output: 2 peluang untuk **12 jam ke depan**: `hujan` dan `cerah`.

Data diambil langsung dari **Open-Meteo Historical Forecast API** (gratis, tanpa key), sumber yang sama dengan yang nanti dipakai web.
Jalankan cell dari atas ke bawah. **Runtime → Change runtime type → T4 GPU**.

In [ ]:
import json, urllib.parse, urllib.request, time
import numpy as np, pandas as pd, keras
from numpy.lib.stride_tricks import sliding_window_view

KOTA = {'Jakarta': (-6.20, 106.85), 'Bekasi': (-6.24, 106.99)}
VARS = ['temperature_2m', 'relative_humidity_2m', 'pressure_msl', 'cloud_cover', 'wind_speed_10m', 'precipitation']
MULAI = '2022-01-01'
AKHIR = str((pd.Timestamp.now() - pd.Timedelta(days=3)).date())
BATAS_UJI = '2025-07-01'  # data setelah tanggal ini cuma buat ujian (nggak dipakai training)
JENDELA, KE_DEPAN = 168, 12  # lihat 168 jam (7 hari) ke belakang, tebak 12 jam ke depan
HUJAN_MM = 0.5    # total hujan 12 jam ke depan >= 0.5 mm -> "hujan"
AWAN_CERAH = 40   # nggak hujan & rata-rata awan < 40% -> "cerah"
EPOCHS = 15

## 1. Ambil data historis tiap kota

In [ ]:
def ambil(lat, lon, **param):
    q = dict(latitude=lat, longitude=lon, hourly=','.join(VARS), timezone='auto', **param)
    url = 'https://historical-forecast-api.open-meteo.com/v1/forecast?' if 'start_date' in q else 'https://api.open-meteo.com/v1/forecast?'
    for coba in range(3):
        try:
            d = json.load(urllib.request.urlopen(url + urllib.parse.urlencode(q), timeout=180))
            break
        except Exception as e:
            print('  gagal, coba lagi:', e); time.sleep(10)
    else:
        raise RuntimeError('Open-Meteo nggak bisa dihubungi')
    df = pd.DataFrame(d['hourly']).set_index('time')
    df.index = pd.to_datetime(df.index)
    df.attrs['utc_offset'] = d['utc_offset_seconds']
    return df.astype('float32').interpolate(limit_direction='both')

data = {}
for nama, (lat, lon) in KOTA.items():
    data[nama] = ambil(lat, lon, start_date=MULAI, end_date=AKHIR)
    print(f'{nama:<11} {len(data[nama]):,} jam')

## 2. Siapkan fitur & label
- **Fitur** (10 angka per jam): 6 variabel cuaca + jam & hari-dalam-tahun (dalam bentuk sin/cos, biar jam 23 dan jam 0 dianggap dekat).
- Hujan pakai `log1p` karena kebanyakan 0 tapi kadang 50 mm.
- **Label** per jam `t`: hujan/cerah di jam `t` s/d `t+11`.

In [ ]:
def fitur(df):
    X = df[VARS].copy()
    X['precipitation'] = np.log1p(X['precipitation'])
    jam, hari = df.index.hour.to_numpy(), df.index.dayofyear.to_numpy()
    X['jam_sin'], X['jam_cos'] = np.sin(2 * np.pi * jam / 24), np.cos(2 * np.pi * jam / 24)
    X['hari_sin'], X['hari_cos'] = np.sin(2 * np.pi * hari / 365.25), np.cos(2 * np.pi * hari / 365.25)
    return X.to_numpy('float32')

def label(df):
    # Y[t] = [hujan, cerah] untuk jam t..t+11
    cs_p = np.concatenate([[0], np.cumsum(df['precipitation'].to_numpy())])
    cs_c = np.concatenate([[0], np.cumsum(df['cloud_cover'].to_numpy())])
    total_hujan = cs_p[KE_DEPAN:] - cs_p[:-KE_DEPAN]
    rata_awan = (cs_c[KE_DEPAN:] - cs_c[:-KE_DEPAN]) / KE_DEPAN
    return np.stack([total_hujan >= HUJAN_MM, (total_hujan < 0.1) & (rata_awan < AWAN_CERAH)], 1).astype('float32')

# Normalisasi pakai rata-rata & simpangan data training saja
latih_semua = np.concatenate([fitur(df[df.index < BATAS_UJI]) for df in data.values()])
MEAN, STD = latih_semua.mean(0), latih_semua.std(0) + 1e-6
FITUR = VARS + ['jam_sin', 'jam_cos', 'hari_sin', 'hari_cos']

def sampel(df, langkah=1):  # 1 = tiap jam jadi sampel (datanya cuma 2 kota)
    X, Y = (fitur(df) - MEAN) / STD, label(df)
    W = sliding_window_view(X, JENDELA, axis=0)          # W[s] = X[s:s+JENDELA] (transpose)
    t = np.arange(JENDELA, len(Y), langkah)               # jam pertama yang ditebak
    uji = df.index[t] >= pd.Timestamp(BATAS_UJI)
    latih = df.index[t + KE_DEPAN - 1] < pd.Timestamp(BATAS_UJI)
    ambil_ = lambda m: (W[t[m] - JENDELA].transpose(0, 2, 1), Y[t[m]], Y[t[m] - KE_DEPAN])
    return ambil_(latih), ambil_(uji)

gabung = lambda bag: [np.concatenate(a) for a in zip(*bag)]
hasil = [sampel(df) for df in data.values()]
X_latih, Y_latih, _ = gabung([h[0] for h in hasil])
X_uji, Y_uji, Y_uji_12jam_lalu = gabung([h[1] for h in hasil])
print('latih:', X_latih.shape, ' uji:', X_uji.shape)
print('proporsi hujan (latih/uji): %.1f%% / %.1f%%' % (Y_latih[:, 0].mean() * 100, Y_uji[:, 0].mean() * 100))
print('proporsi cerah (latih/uji): %.1f%% / %.1f%%' % (Y_latih[:, 1].mean() * 100, Y_uji[:, 1].mean() * 100))

## 3. Training LSTM

In [ ]:
model = keras.Sequential([
    keras.Input((JENDELA, len(FITUR))),
    keras.layers.LSTM(64),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(2, activation='sigmoid'),  # [peluang hujan, peluang cerah]
])
model.compile('adam', 'binary_crossentropy', metrics=[keras.metrics.AUC(name='auc', multi_label=True)])
riwayat = model.fit(X_latih, Y_latih, validation_split=0.15, epochs=EPOCHS, batch_size=256,
                    callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)])

## 4. Ujian jujur: bandingkan dengan tebakan bodoh
Kalau model kalah dari tebakan bodoh, berarti model nggak berguna.
- **Selalu jawab mayoritas**: misal selalu "nggak hujan".
- **Sama kayak 12 jam lalu**: kalau 12 jam terakhir hujan, tebak hujan lagi.

In [ ]:
P = model.predict(X_uji, batch_size=1024, verbose=0)
auc = keras.metrics.AUC()
for i, nama in enumerate(['hujan', 'cerah']):
    y = Y_uji[:, i]
    mayoritas = max(y.mean(), 1 - y.mean())
    persisten = (Y_uji_12jam_lalu[:, i] == y).mean()
    auc.reset_state(); auc.update_state(y, P[:, i])
    print(f'{nama:>5} | akurasi model {((P[:, i] > 0.5) == y).mean():.1%} | '
          f'selalu mayoritas {mayoritas:.1%} | sama kayak 12 jam lalu {persisten:.1%} | AUC model {float(auc.result()):.3f}')

## 5. Coba di data terbaru (cara yang sama dengan web nanti)

In [ ]:
def tebak(lat, lon):
    df = ambil(lat, lon, past_days=8, forecast_days=2)
    sekarang = pd.Timestamp.now('UTC').tz_convert(None) + pd.Timedelta(seconds=df.attrs['utc_offset'])
    lalu, depan = df[df.index <= sekarang], df[df.index > sekarang]
    X = (fitur(lalu.tail(JENDELA)) - MEAN) / STD
    p_hujan, p_cerah = model.predict(X[None], verbose=0)[0]
    om = depan.head(KE_DEPAN)
    print(f'LSTM kita     : hujan {p_hujan:.0%}, cerah {p_cerah:.0%}')
    print(f'Open-Meteo    : total hujan 12 jam {om.precipitation.sum():.1f} mm, rata-rata awan {om.cloud_cover.mean():.0f}%')

for nama in KOTA:
    print('==', nama); tebak(*KOTA[nama])

## 6. Simpan & download
`model_hujan.keras` = model, `hujan_config.json` = urutan fitur + angka normalisasi (wajib dipakai web, kalau beda hasilnya ngawur).

In [ ]:
model.save('model_hujan.keras')
json.dump(dict(fitur=FITUR, mean=MEAN.tolist(), std=STD.tolist(), jendela=JENDELA, ke_depan=KE_DEPAN,
               hujan_mm=HUJAN_MM, awan_cerah=AWAN_CERAH, output=['hujan', 'cerah']),
          open('hujan_config.json', 'w'), indent=1)
from google.colab import files
files.download('model_hujan.keras')
files.download('hujan_config.json')